In [2]:
import pandas as pd
from sklearn.metrics import confusion_matrix

In [4]:
train = pd.read_csv("../data/raw/aps_failure_training_set.csv", skiprows = 20, na_values = "na")

X = train.drop(columns = "class")
y = (train["class"] =="pos").astype(int)

print(X.shape, y.mean().round(4))

(60000, 170) 0.0167


In [5]:
def scania_cost(y_true,y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels = [0, 1]).ravel()
    return 10 * fp + 500 * fn

print("Flag nothing:    ", scania_cost(y,[0] * len(y)))
print("Flag everything: ", scania_cost(y,[1] * len(y)))

Flag nothing:     500000
Flag everything:  590000


In [6]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

dummy = DummyClassifier(strategy = "most_frequent")
pred = cross_val_predict(dummy, X, y, cv = cv)

print("Dummy cost (5-fold CV):", scania_cost(y, pred))

Dummy cost (5-fold CV): 500000


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)

proba = cross_val_predict(logreg, X, y, cv=cv, method="predict_proba")[:, 1]
pred = (proba >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print("Logistic regression cost (5-fold CV):", scania_cost(y, pred))
print(f"Caught {tp} of {tp + fn} failures; {fp} false alarms")

Logistic regression cost (5-fold CV): 63430
Caught 905 of 1000 failures; 1593 false alarms


Logistic regression (median imputation with missing-value flags, scaling, balanced class weights) inside a Pipeline, 5-fold CV: cost 63,430 vs 500,000 for the dummy. Caught 905/1,000 failures with 1,593 false alarms. Missed failures make up 75% of the cost, so the 0.5 threshold is too high for this cost structure.